# C3 — Colectare comentarii YouTube
În acest notebook colectăm un eșantion  de comentarii publice de pe YouTube.
Scopul nu este să obținem corpusul final mare, ci să înțelegem fluxul:
sursă → API → comentarii brute → fișier JSONL.
La final, fiecare student salvează propriul fișier în `data/raw/`.

## 1. Ce trebuie să avem pregătit
Avem nevoie de:
- fișier `.env` în root-ul proiectului
- cheia `YOUTUBE_API_KEY`
- un handle de canal YouTube
Exemplu în `.env`:
```text
YOUTUBE_API_KEY=cheia_ta_aici

In [1]:

from pathlib import Path
import os
import json
import requests
from datetime import datetime
from dotenv import load_dotenv

## 2. Încărcăm cheia API
Notebook-ul caută fișierul `.env` în root-ul proiectului.
Dacă cheia nu este găsită, colectarea nu poate porni.

In [2]:
ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")
API_KEY = os.getenv("YOUTUBE_API_KEY")
BASE_URL = "https://www.googleapis.com/youtube/v3"
print("Root proiect:", ROOT)
print("Cheie găsită:", API_KEY is not None)

Root proiect: c:\Users\Doina\Desktop\ADC\AN 2\8. INGINERIE AI\Proiect\echochamber-app
Cheie găsită: True


## 3. Alegem canalul și numărul de videoclipuri
Fiecare student schimbă `student_id` și `handle`.
Pentru exercițiu folosim puține videoclipuri, ca să nu consumăm inutil cota API.

In [3]:
student_id = "student_03"
handle = "CălinGeorgescu-CanalulOficial"
max_videos = 10
max_comments_per_video = 10
output_file = ROOT / "data" / "raw" / f"{student_id}_youtube_raw.jsonl"
print(output_file)

c:\Users\Doina\Desktop\ADC\AN 2\8. INGINERIE AI\Proiect\echochamber-app\data\raw\student_03_youtube_raw.jsonl


## 4. Găsim canalul YouTube

YouTube lucrează intern cu `channel_id`, nu direct cu numele canalului.
De aceea, primul pas este să transformăm handle-ul în `channel_id`.

In [4]:
channel_response = requests.get(
    f"{BASE_URL}/channels",
    params={
        "part": "id",
        "forHandle": handle,
        "key": API_KEY
    }
)
channel_data = channel_response.json()
channel_data

{'kind': 'youtube#channelListResponse',
 'etag': 'UFTKg9VFdUN4bLLNFOw49HFDgw0',
 'pageInfo': {'totalResults': 1, 'resultsPerPage': 5},
 'items': [{'kind': 'youtube#channel',
   'etag': '0L28yG9EWNGeZg-nQFCSk6LOYwc',
   'id': 'UCOJOwy9IEU6ELUJFHxVFLKQ'}]}

In [5]:
channel_id = channel_data["items"][0]["id"]
channel_id

'UCOJOwy9IEU6ELUJFHxVFLKQ'

## 5. Luăm cele mai recente videoclipuri
Acum cerem ultimele videoclipuri publicate de canal.
Pentru curs folosim doar câteva videoclipuri.

In [6]:
videos_response = requests.get(
    f"{BASE_URL}/search",
    params={
        "part": "snippet",
        "channelId": channel_id,
        "type": "video",
        "order": "date",
        "maxResults": max_videos,
        "key": API_KEY
    }
)
videos_data = videos_response.json()
videos_data["items"][0]

{'kind': 'youtube#searchResult',
 'etag': 'w_ZrE0tE-jWKgatbl_DQnfmgsR4',
 'id': {'kind': 'youtube#video', 'videoId': 'e9dY7rhLG4c'},
 'snippet': {'publishedAt': '2026-04-28T15:44:02Z',
  'channelId': 'UCOJOwy9IEU6ELUJFHxVFLKQ',
  'title': 'Călin Georgescu - România nu are un guvern pe măsura ei ( 28.04.2026 - la ICCJ )',
  'description': '',
  'thumbnails': {'default': {'url': 'https://i.ytimg.com/vi/e9dY7rhLG4c/default.jpg',
    'width': 120,
    'height': 90},
   'medium': {'url': 'https://i.ytimg.com/vi/e9dY7rhLG4c/mqdefault.jpg',
    'width': 320,
    'height': 180},
   'high': {'url': 'https://i.ytimg.com/vi/e9dY7rhLG4c/hqdefault.jpg',
    'width': 480,
    'height': 360}},
  'channelTitle': 'Călin Georgescu - Canalul Oficial',
  'liveBroadcastContent': 'none',
  'publishTime': '2026-04-28T15:44:02Z'}}

In [7]:
videos = []
for item in videos_data["items"]:
    videos.append({
        "video_id": item["id"]["videoId"],
        "video_title": item["snippet"]["title"],
        "video_date": item["snippet"]["publishedAt"][:10]
    })
videos

[{'video_id': 'e9dY7rhLG4c',
  'video_title': 'Călin Georgescu - România nu are un guvern pe măsura ei ( 28.04.2026 - la ICCJ )',
  'video_date': '2026-04-28'},
 {'video_id': 'ki0hPgSlbZ4',
  'video_title': 'Călin Georgescu și maestrul Ion Cristoiu la Realitatea TV ( 16.04.2026 )',
  'video_date': '2026-04-20'},
 {'video_id': 'VD1UoFcNcW0',
  'video_title': 'Călin Georgescu - Ni se interzice să le spunem hoților, hoți ( 2026.04.14 - la ICCJ )',
  'video_date': '2026-04-15'},
 {'video_id': 'HqOScz9dXP4',
  'video_title': 'Călin Georgescu - Sărbătoarea Învierii și unitatea românilor ✝ ( 12.04.2026 )',
  'video_date': '2026-04-11'},
 {'video_id': 'PM9d0b-yUVc',
  'video_title': 'Dragii mei, vă mulțumesc ! ( 09.04.2026 )',
  'video_date': '2026-04-09'},
 {'video_id': 'g9RhhLGdMq0',
  'video_title': 'Călin Georgescu și Anca Alexandrescu la Culisele Statului Paralel ( 07.04.2026 )',
  'video_date': '2026-04-07'},
 {'video_id': 'DG5ox0kWXjA',
  'video_title': 'Călin Georgescu - Adevărul nu se

## 6. Colectăm comentariile
Pentru fiecare videoclip luăm comentariile publice ordonate după relevanță.
În acest exercițiu nu folosim paginare, deci luăm maximum 100 comentarii per videoclip.

In [8]:
comments = []
for video in videos:
    print("Colectez:", video["video_title"][:80])
    comments_response = requests.get(
        f"{BASE_URL}/commentThreads",
        params={
            "part": "snippet",
            "videoId": video["video_id"],
            "maxResults": max_comments_per_video,
            "textFormat": "plainText",
            "order": "relevance",
            "key": API_KEY
        }
    )
    comments_data = comments_response.json()
    for comment_item in comments_data.get("items", []):
        snippet = comment_item["snippet"]["topLevelComment"]["snippet"]
        record = {
            "id": f"yt_{video['video_id']}_{comment_item['id']}",
            "source_platform": "youtube",
            "source_channel": handle,
            "text_raw": snippet["textDisplay"],
            "video_id": video["video_id"],
            "video_title": video["video_title"],
            "video_date": video["video_date"],
            "comment_date": snippet["publishedAt"][:10],
            "likes": snippet["likeCount"],
            "collected_at": datetime.utcnow().strftime("%Y-%m-%d")
        }
        comments.append(record)
len(comments)

Colectez: Călin Georgescu - România nu are un guvern pe măsura ei ( 28.04.2026 - la ICCJ )
Colectez: Călin Georgescu și maestrul Ion Cristoiu la Realitatea TV ( 16.04.2026 )
Colectez: Călin Georgescu - Ni se interzice să le spunem hoților, hoți ( 2026.04.14 - la I
Colectez: Călin Georgescu - Sărbătoarea Învierii și unitatea românilor ✝ ( 12.04.2026 )
Colectez: Dragii mei, vă mulțumesc ! ( 09.04.2026 )
Colectez: Călin Georgescu și Anca Alexandrescu la Culisele Statului Paralel ( 07.04.2026 )
Colectez: Călin Georgescu - Adevărul nu se afirmă, se dovedește ( 21.01.2026 - la Curtea d
Colectez: Călin Georgescu - Dumnezeu veghează asupra poporului român ( 26.03.2026 - Judeca
Colectez: Călin Georgescu - Sădirea pomilor ( 14.03.2026 )
Colectez: Călin Georgescu - Ziua sădirii arborelui ( 10.03.2026 - la Poliția Buftea )


100

# Explorare si curatare

## 7. Inspectăm primele comentarii
Înainte să salvăm fișierul, verificăm dacă datele arată cum trebuie.

In [9]:
comments[:3]

[{'id': 'yt_e9dY7rhLG4c_Ugy5iL_s7fKRtJfNPlx4AaABAg',
  'source_platform': 'youtube',
  'source_channel': 'CălinGeorgescu-CanalulOficial',
  'text_raw': 'Felicitāri tuturor celor prezenti inclusiv reporterilor!',
  'video_id': 'e9dY7rhLG4c',
  'video_title': 'Călin Georgescu - România nu are un guvern pe măsura ei ( 28.04.2026 - la ICCJ )',
  'video_date': '2026-04-28',
  'comment_date': '2026-04-28',
  'likes': 24,
  'collected_at': '2026-05-05'},
 {'id': 'yt_e9dY7rhLG4c_UgyeLBTqmuedi83PHqF4AaABAg',
  'source_platform': 'youtube',
  'source_channel': 'CălinGeorgescu-CanalulOficial',
  'text_raw': 'Hristos a inviat!',
  'video_id': 'e9dY7rhLG4c',
  'video_title': 'Călin Georgescu - România nu are un guvern pe măsura ei ( 28.04.2026 - la ICCJ )',
  'video_date': '2026-04-28',
  'comment_date': '2026-04-28',
  'likes': 53,
  'collected_at': '2026-05-05'},
 {'id': 'yt_e9dY7rhLG4c_Ugy0IlTDuSjSxQl__tx4AaABAg',
  'source_platform': 'youtube',
  'source_channel': 'CălinGeorgescu-CanalulOficial

In [10]:
comments[0].keys()

dict_keys(['id', 'source_platform', 'source_channel', 'text_raw', 'video_id', 'video_title', 'video_date', 'comment_date', 'likes', 'collected_at'])

## 8. Curățare minimă a textului
Acum pornim de la `text_raw` și construim o variantă curățată în câmpul `text`.
Nu schimbăm sensul comentariului. Eliminăm doar zgomot simplu: linkuri, spații inutile, texte prea scurte și duplicate.

In [11]:
import re

def clean_text(text):
    text = re.sub(r"http\S+", "", text)      # elimină linkuri
    text = re.sub(r"\s+", " ", text)         # normalizează spațiile
    return text.strip()

## 9. Aplicăm curățarea
Pentru fiecare comentariu păstrăm textul original în `text_raw` și adăugăm textul curățat în `text`.

In [13]:
for comment in comments:
    comment["text"] = clean_text(comment["text_raw"])

comments[0]

{'id': 'yt_e9dY7rhLG4c_Ugy5iL_s7fKRtJfNPlx4AaABAg',
 'source_platform': 'youtube',
 'source_channel': 'CălinGeorgescu-CanalulOficial',
 'text_raw': 'Felicitāri tuturor celor prezenti inclusiv reporterilor!',
 'video_id': 'e9dY7rhLG4c',
 'video_title': 'Călin Georgescu - România nu are un guvern pe măsura ei ( 28.04.2026 - la ICCJ )',
 'video_date': '2026-04-28',
 'comment_date': '2026-04-28',
 'likes': 24,
 'collected_at': '2026-05-05',
 'text': 'Felicitāri tuturor celor prezenti inclusiv reporterilor!'}

## 10. Filtrăm comentariile prea scurte
Pentru exercițiu păstrăm doar comentariile care au cel puțin 60 de caractere.
Comentariile foarte scurte sunt greu de interpretat în analiza discursivă.

In [14]:
MIN_CHARS = 60

comments_clean = [
    comment for comment in comments
    if len(comment["text"]) >= MIN_CHARS
]

print("Comentarii brute:", len(comments))
print("Comentarii după filtrarea lungimii:", len(comments_clean))

Comentarii brute: 100
Comentarii după filtrarea lungimii: 58


## 11. Filtrăm textele cu prea puține litere
Comentariile formate mai ales din emoji, simboluri sau caractere izolate produc zgomot.
Păstrăm comentariile în care cel puțin 50% dintre caractere sunt litere.

In [15]:
MIN_ALPHA = 0.5

def alpha_ratio(text):
    if len(text) == 0:
        return 0
    letters = sum(char.isalpha() for char in text)
    return letters / len(text)

comments_clean = [
    comment for comment in comments_clean
    if alpha_ratio(comment["text"]) >= MIN_ALPHA
]

print("Comentarii după filtrarea literelor:", len(comments_clean))

Comentarii după filtrarea literelor: 58


## 12. Eliminăm duplicatele
Dacă același text apare de mai multe ori, îl păstrăm o singură dată.

In [16]:
seen_texts = set()
unique_comments = []

for comment in comments_clean:
    text = comment["text"].lower()
    if text not in seen_texts:
        unique_comments.append(comment)
        seen_texts.add(text)

comments_clean = unique_comments

print("Comentarii finale după deduplicare:", len(comments_clean))

Comentarii finale după deduplicare: 57


## 14. Salvăm fișierul curățat
Salvăm rezultatul în `data/cleaned/`.

In [17]:
clean_output_file = ROOT / "data" / "cleaned" / f"{student_id}_youtube_clean.jsonl"
clean_output_file.parent.mkdir(parents=True, exist_ok=True)

with clean_output_file.open("w", encoding="utf-8") as f:
    for comment in comments_clean:
        f.write(json.dumps(comment, ensure_ascii=False) + "\n")

print("Comentarii curate salvate:", len(comments_clean))
print("Fișier:", clean_output_file)

Comentarii curate salvate: 57
Fișier: c:\Users\Doina\Desktop\ADC\AN 2\8. INGINERIE AI\Proiect\echochamber-app\data\cleaned\student_03_youtube_clean.jsonl


# Functia de curatare

In [18]:
import re

def clean_comments(comments, min_chars=60, min_alpha=0.5):
    cleaned = []
    seen_texts = set()
    
    for comment in comments:
        # 1. Curățare text
        text = comment["text_raw"]
        text = re.sub(r"http\S+", "", text)
        text = re.sub(r"\s+", " ", text).strip()
        
        # 2. Filtru lungime
        if len(text) < min_chars:
            continue
        
        # 3. Filtru proporție litere
        letters = sum(char.isalpha() for char in text)
        alpha_ratio = letters / len(text) if len(text) > 0 else 0
        
        if alpha_ratio < min_alpha:
            continue
        
        # 4. Filtru duplicate
        text_key = text.lower()
        if text_key in seen_texts:
            continue
        
        seen_texts.add(text_key)
        
        # 5. Păstrăm comentariul și adăugăm textul curățat
        new_comment = comment.copy()
        new_comment["text"] = text
        new_comment["lang"] = "ro"
        cleaned.append(new_comment)
    
    return cleaned

In [19]:
comments_clean = clean_comments(
    comments,
    min_chars=60,
    min_alpha=0.5
)

print("Comentarii brute:", len(comments))
print("Comentarii curate:", len(comments_clean))

Comentarii brute: 100
Comentarii curate: 57


In [20]:
for comment in comments_clean[:3]:
    print("RAW:", comment["text_raw"])
    print("CLEAN:", comment["text"])
    print("---")

RAW: Hristos A Înviat🙏Să fiți binecuvântați toți care ați fost alături de Președintele nostru!
Noi de aici din străinătate sunte alături de voi cu tot sufletul.
Doamne Ajută🙏❤️
CLEAN: Hristos A Înviat🙏Să fiți binecuvântați toți care ați fost alături de Președintele nostru! Noi de aici din străinătate sunte alături de voi cu tot sufletul. Doamne Ajută🙏❤️
---
RAW: Să ne trăiți d-le președinte CĂLIN GEORGESCU 
Impreuna vom face istorie, nu renunțăm până nu ne luăm țara înapoi!
CLEAN: Să ne trăiți d-le președinte CĂLIN GEORGESCU Impreuna vom face istorie, nu renunțăm până nu ne luăm țara înapoi!
---
RAW: Președintele nostru adevărat ! Bravo domnule Calin ! Tot respectul
CLEAN: Președintele nostru adevărat ! Bravo domnule Calin ! Tot respectul
---


In [21]:
clean_output_file = ROOT / "data" / "cleaned" / f"{student_id}_youtube_clean.jsonl"
clean_output_file.parent.mkdir(parents=True, exist_ok=True)

with clean_output_file.open("w", encoding="utf-8") as f:
    for comment in comments_clean:
        f.write(json.dumps(comment, ensure_ascii=False) + "\n")

print("Fișier salvat:", clean_output_file)
print("Comentarii salvate:", len(comments_clean))

Fișier salvat: c:\Users\Doina\Desktop\ADC\AN 2\8. INGINERIE AI\Proiect\echochamber-app\data\cleaned\student_03_youtube_clean.jsonl
Comentarii salvate: 57


15. Ce am obținut
Am produs două fișiere:
- `data/raw/student_XX_youtube_raw.jsonl` — comentarii brute
- `data/cleaned/student_XX_youtube_clean.jsonl` — comentarii curățate
Fișierul curățat va putea fi unit cu fișierele celorlalți membri ai echipei.